## Transformer Model DistilBERT
---

In [1]:
!pip install transformers datasets accelerate evaluate

  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 27.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 28.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 24.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 23.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 21.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.1 MB/s  0:00:00m0:00:01
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: setuptools━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/32 [sympy

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [5]:
# loading the dataset and removing the unusable text

DATA_PATH = Path("../data/processed/WELFake_processed.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df[["text", "label"]].head())

df = df.dropna(subset=["text", "label"]).copy()

df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

print("Dataset shape after cleaning:", df.shape)


Dataset shape: (62704, 14)
                                                text  label
0  No comment is expected from Barack Obama Membe...      1
1     Did they post their votes for Hillary already?      1
2  Now, most of the demonstrators gathered last n...      1
3  A dozen politically active pastors came here f...      0
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1
Dataset shape after cleaning: (62704, 14)


In [6]:
# Separating the dataset into train, validation, and test sets

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 45146
Validation: 5017
Testing: 12541


In [7]:
# converting the pandas dataframes into Hugging Face datasets

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

In [8]:
# Load DistilBERT tokenizer 

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Python(49152) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Map:   0%|          | 0/45146 [00:00<?, ? examples/s]

Map:   0%|          | 0/5017 [00:00<?, ? examples/s]

Map:   0%|          | 0/12541 [00:00<?, ? examples/s]

In [9]:
# Load the model and the evaluation function

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Training settings 
training_args = TrainingArguments(
    output_dir="../data/transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [13]:
# Trainer initialization

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [14]:
# Train it

trainer.train()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.040451,0.037095,0.990433,0.995047,0.983534,0.989257
2,0.018863,0.038034,0.992625,0.996406,0.987094,0.991728


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11288, training_loss=0.04932859280192201, metrics={'train_runtime': 18625.2883, 'train_samples_per_second': 4.848, 'train_steps_per_second': 0.606, 'total_flos': 5980373179723776.0, 'train_loss': 0.04932859280192201, 'epoch': 2.0})

In [15]:
# Evaluate on the validation test set
val_results = trainer.evaluate()
print(val_results)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.018863,0.038034,2,0.992625,0.996406,0.987094,0.991728


{'eval_loss': 0.03803360462188721, 'eval_accuracy': 0.992625074745864, 'eval_precision': 0.9964061096136568, 'eval_recall': 0.9870939029817535, 'eval_f1': 0.9917281466577241}


In [16]:
# Evalutate on the test set
test_results = trainer.evaluate(eval_dataset=test_tokenized)
print(test_results)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.018863,0.043360,2,0.991707,0.991793,0.989674,0.990732


{'eval_loss': 0.04335964471101761, 'eval_accuracy': 0.9917072003827446, 'eval_precision': 0.991793041926851, 'eval_recall': 0.9896742033113762, 'eval_f1': 0.9907324897522723}


In [17]:
comparison = pd.DataFrame({
    "Model": [
        "Linear SVM",
        "DistilBERT"
    ],
    "Accuracy": [
        0.966829,
        test_results["eval_accuracy"]
    ],
    "Precision": [
        0.965622,
        test_results["eval_precision"]
    ],
    "Recall": [
        0.960121,
        test_results["eval_recall"]
    ],
    "F1 Score": [
        0.962864,
        test_results["eval_f1"]
    ]
})

display(comparison)

,Model,Accuracy,Precision,Recall,F1 Score
0,Linear SVM,0.966829,0.965622,0.960121,0.962864
1,DistilBERT,0.991707,0.991793,0.989674,0.990732


In [18]:
trainer.save_model("../data/transformer_model")
tokenizer.save_pretrained("../data/transformer_model")

print("Model and tokenizer saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved!


In [19]:
test_results = trainer.evaluate(
    eval_dataset=test_tokenized
)

print(test_results)

pd.DataFrame([test_results]).to_csv(
    "../data/transformer_test_results.csv",
    index=False
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.018863,0.043360,2,0.991707,0.991793,0.989674,0.990732


{'eval_loss': 0.04335964471101761, 'eval_accuracy': 0.9917072003827446, 'eval_precision': 0.991793041926851, 'eval_recall': 0.9896742033113762, 'eval_f1': 0.9907324897522723}


In [20]:
transformer_output = trainer.predict(test_tokenized)

transformer_predictions = transformer_output.predictions.argmax(axis=1)

predictions_df = test_df.copy()
predictions_df["predicted_label"] = transformer_predictions

predictions_df.to_csv(
    "../data/transformer_predictions.csv",
    index=False
)

print("Predictions saved!")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Predictions saved!
